# Velocity Pushforward Table 

Minimal notebook to reproduce the comparison table for CFM, MFM, UOT, and SF2M velocity pushforward methods.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict

import pandas as pd
try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None


FLOW_METRICS_PATTERN = 'alpha_*/fm/seed_*/sample_velocity/metrics.json'
METHOD_ORDER = ['CFM', 'MFM', 'UOT', 'SF2M']

FLOW_ARTIFACT_DIRS = {
    'light': {
        'CFM': Path('/mnt/data/nvth2.data/experiments/light/align_sweep/2025-09-21/19-04-26/artifacts'),
        'MFM': Path('/mnt/data/nvth2.data/experiments/light/align_sweep/2025-09-21/19-04-26_mfm/artifacts'),
        'UOT': Path('/mnt/data/nvth2.data/experiments/light/align_sweep/2025-11-16/16-44-39_uot/artifacts'),
        'SF2M': Path('/mnt/data/nvth2.data/experiments/light/align_sweep/2025-09-21/19-04-26_sf2m/artifacts'),
    },
    'immune': {
        'CFM': Path('/mnt/data/nvth2.data/experiments/immune/align_sweep/2025-09-21/21-34-03/artifacts'),
        'MFM': Path('/mnt/data/nvth2.data/experiments/immune/align_sweep/2025-09-21/21-34-03_mfm/artifacts'),
        'UOT': Path('/mnt/data/nvth2.data/experiments/immune/align_sweep/2025-11-16/16-21-48_uot/artifacts'),
        'SF2M': Path('/mnt/data/nvth2.data/experiments/immune/align_sweep/2025-09-21/21-34-03_sf2m/artifacts'),
    },
    'cancer': {
        'CFM': Path('/mnt/data/nvth2.data/experiments/cancer/align_sweep/2025-09-21/22-27-38/artifacts'),
        'MFM': Path('/mnt/data/nvth2.data/experiments/cancer/align_sweep/2025-09-21/22-27-38_mfm/artifacts'),
        'UOT': Path('/mnt/data/nvth2.data/experiments/cancer/align_sweep/2025-11-16/16-44-24_uot/artifacts'),
        'SF2M': Path('/mnt/data/nvth2.data/experiments/cancer/align_sweep/2025-09-21/22-27-38_sf2m/artifacts'),
    },
}

DATASETS = list(FLOW_ARTIFACT_DIRS.keys())


In [2]:
def _flatten_dict(d: Dict, prefix: str = '') -> Dict:
    flat = {}
    for k, v in d.items():
        if isinstance(v, dict):
            flat.update(_flatten_dict(v, prefix=f'{prefix}{k}.'))
        else:
            flat[f'{prefix}{k}'] = v
    return flat


def load_flow_matching_metrics(
    flow_artifact_dirs: Dict[str, Dict[str, Path]],
    pattern: str = FLOW_METRICS_PATTERN,
) -> pd.DataFrame:
    rows = []

    for dataset, method_to_dir in flow_artifact_dirs.items():
        for method, artifacts_dir in method_to_dir.items():
            artifacts_dir = Path(artifacts_dir)
            if not artifacts_dir.exists():
                raise FileNotFoundError(f'Missing artifacts dir for {dataset}/{method}: {artifacts_dir}')

            metrics_files = sorted(artifacts_dir.glob(pattern))
            if not metrics_files:
                raise RuntimeError(
                    f'No metrics files found for {dataset}/{method} under {artifacts_dir} with pattern {pattern}'
                )

            timestamp = artifacts_dir.parent.name
            for metrics_file in metrics_files:
                parts = metrics_file.parts
                alpha = next((part.replace('alpha_', '') for part in parts if part.startswith('alpha_')), None)
                seed = next((part.replace('seed_', '') for part in parts if part.startswith('seed_')), None)

                try:
                    with metrics_file.open('r') as f:
                        metrics = json.load(f)
                except Exception as e:
                    print(f'Failed to read {metrics_file}: {e}')
                    continue

                record = {
                    'path': str(metrics_file),
                    'dataset': dataset,
                    'alpha': alpha,
                    'seed': seed,
                    'method': method,
                    'timestamp': timestamp,
                }
                record.update(_flatten_dict(metrics if isinstance(metrics, dict) else {'value': metrics}))
                rows.append(record)

    df = pd.DataFrame(rows)
    if not df.empty:
        df['alpha'] = pd.to_numeric(df['alpha'], errors='coerce')
        df['seed'] = df['seed'].astype(str)
    return df


def format_meanstd(series_like: pd.Series, is_best: bool = False) -> str:
    s = pd.to_numeric(series_like, errors='coerce').dropna()
    if s.empty:
        return '-'
    mean = float(s.mean())
    std = float(s.std()) if not pd.isna(s.std()) else 0.0
    cell = f'\\meanstd{{{mean:.3f}}}{{{std:.3f}}}'
    if is_best:
        return f'\\textbf{{\\boldmath {cell}}}'
    return cell


In [3]:
if not FLOW_ARTIFACT_DIRS:
    raise RuntimeError('Set FLOW_ARTIFACT_DIRS first: one artifacts folder per dataset/method.')

for dataset, method_to_dir in FLOW_ARTIFACT_DIRS.items():
    if not method_to_dir:
        raise RuntimeError(f'No methods configured for dataset: {dataset}')

    missing_methods = [m for m in METHOD_ORDER if m not in method_to_dir]
    if missing_methods:
        raise RuntimeError(f'Missing methods for dataset {dataset}: {missing_methods}')

    for method, artifacts_dir in method_to_dir.items():
        artifacts_dir = Path(artifacts_dir)
        if not artifacts_dir.exists():
            raise FileNotFoundError(f'Missing artifacts dir for {dataset}/{method}: {artifacts_dir}')

df = load_flow_matching_metrics(FLOW_ARTIFACT_DIRS)

if df.empty:
    raise RuntimeError('No flow-matching metrics loaded from FLOW_ARTIFACT_DIRS.')

# Keep only configured datasets and ordered methods for table generation.
df = df[df['dataset'].isin(DATASETS)].copy()
df = df[df['method'].isin(METHOD_ORDER)].copy()

print(f'Flow-matching rows: {len(df)}')
print('Methods:', sorted(df['method'].unique()))
print('Alphas:', sorted(df['alpha'].dropna().unique()))
print('Datasets:', DATASETS)


Flow-matching rows: 180
Methods: ['CFM', 'MFM', 'SF2M', 'UOT']
Alphas: [np.float64(0.0), np.float64(0.5), np.float64(1.0)]
Datasets: ['light', 'immune', 'cancer']


In [4]:
w1_metric = 'pushforward.wasserstein_1'
w2_metric = 'pushforward.wasserstein_2'

if w1_metric not in df.columns or w2_metric not in df.columns:
    raise RuntimeError(f'Missing required metrics columns: {w1_metric}, {w2_metric}')

alphas = sorted(df['alpha'].dropna().unique())
methods = [m for m in METHOD_ORDER if m in set(df['method'].unique())]

latex = []
latex.append('\\begin{table}')
latex.append('\\centering')
latex.append('\\caption{Velocity Pushforward Results by Dataset and $\\alpha$ Value}')
latex.append('\\label{tab:flow_matching_results}')
latex.append('\\begin{tabular}{ll' + 'cc' * len(DATASETS) + '}')
latex.append('\\toprule')

dataset_header = '& & ' + ' & '.join([f'\\multicolumn{{2}}{{c}}{{\\textbf{{{d}}}}}' for d in DATASETS]) + ' \\\\'
latex.append(dataset_header)

metric_header = 'Method & $\\alpha$ & ' + ' & '.join(['W$_1$ & W$_2$' for _ in DATASETS]) + ' \\\\'
latex.append(metric_header)
latex.append('\\midrule')

# Determine best (minimum mean) per dataset/metric across configured methods.
best_w1 = {}
best_w2 = {}
for dataset in DATASETS:
    candidates_w1 = []
    candidates_w2 = []

    for method in methods:
        for alpha in alphas:
            subset = df[(df['dataset'] == dataset) & (df['method'] == method) & (df['alpha'] == alpha)]
            if not subset.empty:
                w1 = pd.to_numeric(subset[w1_metric], errors='coerce').dropna()
                w2 = pd.to_numeric(subset[w2_metric], errors='coerce').dropna()
                if not w1.empty:
                    candidates_w1.append(float(w1.mean()))
                if not w2.empty:
                    candidates_w2.append(float(w2.mean()))

    if candidates_w1:
        best_w1[dataset] = min(candidates_w1)
    if candidates_w2:
        best_w2[dataset] = min(candidates_w2)

for method in methods:
    method_df = df[df['method'] == method]
    method_alphas = [a for a in alphas if a in set(method_df['alpha'].unique())]

    for idx, alpha in enumerate(method_alphas):
        alpha_label = f'{float(alpha):.3f}'
        if idx == 0:
            row = f'\\multirow{{{len(method_alphas)}}}{{*}}{{{method}}} & {alpha_label} & '
        else:
            row = f'& {alpha_label} & '

        for dataset in DATASETS:
            subset = method_df[(method_df['alpha'] == alpha) & (method_df['dataset'] == dataset)]
            if subset.empty:
                row += '- & - & '
                continue

            w1_series = pd.to_numeric(subset[w1_metric], errors='coerce')
            w2_series = pd.to_numeric(subset[w2_metric], errors='coerce')
            w1_mean = float(w1_series.dropna().mean()) if not w1_series.dropna().empty else None
            w2_mean = float(w2_series.dropna().mean()) if not w2_series.dropna().empty else None

            is_best_w1 = w1_mean is not None and dataset in best_w1 and abs(w1_mean - best_w1[dataset]) < 1e-9
            is_best_w2 = w2_mean is not None and dataset in best_w2 and abs(w2_mean - best_w2[dataset]) < 1e-9

            row += f"{format_meanstd(w1_series, is_best_w1)} & {format_meanstd(w2_series, is_best_w2)} & "

        latex.append(row.rstrip('& ') + ' \\\\')

latex.append('\\bottomrule')
latex.append('\\end{tabular}')
latex.append('\\end{table}')

latex_code = '\n'.join(latex)

print(latex_code)
if display is not None and Markdown is not None:
    display(Markdown(f'```latex\n{latex_code}\n```'))


\begin{table}
\centering
\caption{Velocity Pushforward Results by Dataset and $\alpha$ Value}
\label{tab:flow_matching_results}
\begin{tabular}{llcccccc}
\toprule
& & \multicolumn{2}{c}{\textbf{light}} & \multicolumn{2}{c}{\textbf{immune}} & \multicolumn{2}{c}{\textbf{cancer}} \\
Method & $\alpha$ & W$_1$ & W$_2$ & W$_1$ & W$_2$ & W$_1$ & W$_2$ \\
\midrule
\multirow{3}{*}{CFM} & 0.000 & \meanstd{2.392}{0.005} & \meanstd{2.625}{0.007} & \meanstd{3.696}{0.007} & \meanstd{3.857}{0.009} & \meanstd{1.993}{0.004} & \meanstd{2.275}{0.005} \\
& 0.500 & \meanstd{2.381}{0.004} & \meanstd{2.618}{0.003} & \meanstd{3.679}{0.009} & \meanstd{3.835}{0.010} & \meanstd{1.989}{0.004} & \textbf{\boldmath \meanstd{2.272}{0.005}} \\
& 1.000 & \meanstd{2.362}{0.003} & \textbf{\boldmath \meanstd{2.601}{0.005}} & \meanstd{3.639}{0.021} & \meanstd{3.788}{0.021} & \meanstd{2.057}{0.005} & \meanstd{2.329}{0.005} \\
\multirow{3}{*}{MFM} & 0.000 & \meanstd{2.401}{0.003} & \meanstd{2.636}{0.003} & \meanstd{3.714}{0.

```latex
\begin{table}
\centering
\caption{Velocity Pushforward Results by Dataset and $\alpha$ Value}
\label{tab:flow_matching_results}
\begin{tabular}{llcccccc}
\toprule
& & \multicolumn{2}{c}{\textbf{light}} & \multicolumn{2}{c}{\textbf{immune}} & \multicolumn{2}{c}{\textbf{cancer}} \\
Method & $\alpha$ & W$_1$ & W$_2$ & W$_1$ & W$_2$ & W$_1$ & W$_2$ \\
\midrule
\multirow{3}{*}{CFM} & 0.000 & \meanstd{2.392}{0.005} & \meanstd{2.625}{0.007} & \meanstd{3.696}{0.007} & \meanstd{3.857}{0.009} & \meanstd{1.993}{0.004} & \meanstd{2.275}{0.005} \\
& 0.500 & \meanstd{2.381}{0.004} & \meanstd{2.618}{0.003} & \meanstd{3.679}{0.009} & \meanstd{3.835}{0.010} & \meanstd{1.989}{0.004} & \textbf{\boldmath \meanstd{2.272}{0.005}} \\
& 1.000 & \meanstd{2.362}{0.003} & \textbf{\boldmath \meanstd{2.601}{0.005}} & \meanstd{3.639}{0.021} & \meanstd{3.788}{0.021} & \meanstd{2.057}{0.005} & \meanstd{2.329}{0.005} \\
\multirow{3}{*}{MFM} & 0.000 & \meanstd{2.401}{0.003} & \meanstd{2.636}{0.003} & \meanstd{3.714}{0.008} & \meanstd{3.880}{0.009} & \meanstd{1.984}{0.004} & \meanstd{2.285}{0.004} \\
& 0.500 & \meanstd{2.393}{0.007} & \meanstd{2.631}{0.008} & \meanstd{3.679}{0.007} & \meanstd{3.838}{0.009} & \meanstd{1.978}{0.004} & \meanstd{2.277}{0.003} \\
& 1.000 & \meanstd{2.363}{0.002} & \meanstd{2.606}{0.002} & \meanstd{3.668}{0.010} & \meanstd{3.824}{0.011} & \meanstd{2.013}{0.003} & \meanstd{2.304}{0.003} \\
\multirow{3}{*}{UOT} & 0.000 & \meanstd{2.411}{0.005} & \meanstd{2.649}{0.006} & \meanstd{3.701}{0.006} & \meanstd{3.867}{0.007} & \meanstd{1.998}{0.004} & \meanstd{2.348}{0.004} \\
& 0.500 & \meanstd{2.377}{0.004} & \meanstd{2.619}{0.005} & \meanstd{3.688}{0.012} & \meanstd{3.854}{0.012} & \textbf{\boldmath \meanstd{1.971}{0.005}} & \meanstd{2.322}{0.005} \\
& 1.000 & \textbf{\boldmath \meanstd{2.360}{0.002}} & \meanstd{2.605}{0.001} & \textbf{\boldmath \meanstd{3.624}{0.004}} & \textbf{\boldmath \meanstd{3.780}{0.002}} & \meanstd{1.993}{0.004} & \meanstd{2.335}{0.005} \\
\multirow{3}{*}{SF2M} & 0.000 & \meanstd{3.254}{0.192} & \meanstd{3.368}{0.182} & \meanstd{4.333}{0.279} & \meanstd{4.436}{0.282} & \meanstd{3.826}{0.265} & \meanstd{3.974}{0.308} \\
& 0.500 & \meanstd{3.199}{0.117} & \meanstd{3.315}{0.110} & \meanstd{4.303}{0.213} & \meanstd{4.397}{0.205} & \meanstd{3.809}{0.302} & \meanstd{3.968}{0.374} \\
& 1.000 & \meanstd{3.226}{0.075} & \meanstd{3.339}{0.073} & \meanstd{4.289}{0.110} & \meanstd{4.387}{0.108} & \meanstd{3.638}{0.308} & \meanstd{3.739}{0.335} \\
\bottomrule
\end{tabular}
\end{table}
```